## IMPORTS

In [22]:
import os
from datetime import date

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from scipy.spatial import cKDTree
import warnings
from scipy.sparse import coo_matrix, identity
from scipy.sparse.linalg import spsolve


import stable_baselines3, sb3_contrib

In [15]:
# ── PDE:  u_t + c u_x = alpha u_xx   on x in [0, 1],  u(0,t) = u(1,t) = 0 ──
alpha = 0.1          # diffusion coefficient
c     = 1.0          # advection speed

# ── Grid: every point the map can use sits on this grid ──
nx = 100
T  = 0.5
dx = 1.0 / (nx - 1)
dt_cfl = min(0.9 * dx / c, 0.45 * dx**2 / alpha)     # advective and diffusive limits
nt = max(int(np.ceil(T / dt_cfl)) + 1, 10)
dt = T / (nt - 1)
# ── Exact solution ──
# u = exp(b x - alpha b^2 t) v  with  b = c / (2 alpha)  removes the advection term;
# v solves the heat equation, so it is a sine series.
_xq = np.linspace(0.0, 1.0, 80001)
_b  = c / (2 * alpha)
u0  = lambda x: np.sin(np.pi * x) + 0.5 * np.sin(2 * np.pi * x)
_v0 = u0(_xq) * np.exp(-_b * _xq)
_n  = np.arange(1, 801)
_bn = np.array([2 * np.trapezoid(_v0 * np.sin(k * np.pi * _xq), _xq) for k in _n])

def u_true(x, t):
    """Exact u at positions x (scalar or array) and one time t. Always returns an array."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    v = (np.sin(np.pi * np.outer(x, _n)) * (_bn * np.exp(-alpha * (_n * np.pi)**2 * t))).sum(axis=1)
    return np.exp(_b * x - alpha * _b**2 * t) * v

In [16]:
# ── FD reference: the tight central stencil (k=1, W=1, s=0) at every point ──
r, nu = alpha * dt / dx**2, c * dt / dx
W_FD = np.array([r + nu / 2, 1 - 2 * r, r - nu / 2])      # weights for (x-dx, x, x+dx)
_fd_cache = {}
def fd_reference(ix, it):
    """FD at z* = (ix, it): (u_fd, E_FD, C_FD). Cached in a dict so the env can be sent to subprocesses."""
    if (ix, it) in _fd_cache:
        return _fd_cache[(ix, it)]
    u = u_true(np.arange(nx) * dx, 0.0)                    # level 0: initial condition
    for _ in range(it):
        u[1:-1] = W_FD[0] * u[:-2] + W_FD[1] * u[1:-1] + W_FD[2] * u[2:]
        u[0] = u[-1] = 0.0                                   # boundaries
    u_fd = u[ix]
    E_FD = abs(u_fd - u_true(ix * dx, it * dt)[0])
    C_FD = sum(min(nx - 2, ix + j) - max(1, ix - j) + 1 for j in range(it))
    _fd_cache[(ix, it)] = (u_fd, E_FD, C_FD)
    return _fd_cache[(ix, it)]

In [ ]:
COND_MAX = 1e4
W_MAX    = 2     # cap on ||w||_1, the per-level error-amplification factor (thesis Thm 2.7)

def solve_weights(dxs, dts, w_max=W_MAX):
    """Min-norm weights for the 3 PDE-substituted Taylor rows (constant, u_x, u_xx).
    None if rank-deficient, ill-conditioned (cond > COND_MAX), or ||w||_1 > w_max."""
    h = max(np.abs(dxs).max(), np.sqrt(alpha * np.abs(dts).max()))
    A = np.array([np.ones_like(dxs),
                  (dxs - c * dts) / h,
                  (0.5 * (dxs - c * dts)**2 + alpha * dts) / h**2])
    if np.linalg.matrix_rank(A) < 3 or np.linalg.cond(A) > COND_MAX:
        return None
    w = np.linalg.pinv(A) @ np.array([1.0, 0.0, 0.0])
    return w if np.abs(w).sum() <= w_max else None

In [19]:
class Cloud:
    """Every point computed or known so far, as grid indices (ix, it).
    x/dx = ix and t/dt = it exactly, so nearest-neighbour in (ix, it) IS the normalised distance."""
    def __init__(self):
        self.pts = []
        self._idx = {}
        self._tree = None
        self._dirty = True

    def add(self, ix, it):
        if (ix, it) in self._idx:
            return
        self._idx[(ix, it)] = len(self.pts)
        self.pts.append((ix, it))
        self._dirty = True

    def __contains__(self, p):
        return p in self._idx

    def neighbours(self, ix, it, k):
        """k nearest cloud points to (ix, it), any direction, excluding (ix, it) itself."""
        if self._dirty:
            self._tree = cKDTree(np.array(self.pts, dtype=float))
            self._dirty = False
        dist, idx = self._tree.query([ix, it], k=min(k + 1, len(self.pts)))
        idx = np.atleast_1d(idx)
        out = [self.pts[i] for i in idx if self.pts[i] != (ix, it)]
        return out[:k]

In [21]:
U0 = u_true(np.arange(nx) * dx, 0.0)      # initial condition on the grid

def is_known(ix, it):
    """Initial condition or boundary: the value is given, not solved for."""
    return it == 0 or ix == 0 or ix == nx - 1

def known_value(ix, it):
    return U0[ix] if it == 0 else 0.0

def build_stencil(cloud, ix, it, k=5):
    """k-NN stencil for (ix, it) from the current cloud. None if inadmissible."""
    nbs = cloud.neighbours(ix, it, k)
    dxs = np.array([q[0] - ix for q in nbs]) * dx
    dts = np.array([q[1] - it for q in nbs]) * dt
    w = solve_weights(dxs, dts)
    return None if w is None else (nbs, w)


In [24]:
def evaluate(cloud, k=5):
    """Solve (I - W) u = b for every non-known point in the cloud, via fresh k-NN stencils.
    Returns (u_hat dict, stencils dict), or (None, None) if any point lacks an admissible
    stencil or the system is singular."""
    pts = [p for p in cloud.pts if not is_known(*p)]
    stencils = {}
    for p in pts:
        st = build_stencil(cloud, *p, k=k)
        if st is None:
            return None, None
        stencils[p] = st

    idx = {p: i for i, p in enumerate(pts)}
    rows, cols, vals, b = [], [], [], np.zeros(len(pts))
    for p, (nbs, w) in stencils.items():
        for q, wq in zip(nbs, w):
            if is_known(*q):
                b[idx[p]] += wq * known_value(*q)
            else:
                rows.append(idx[p]); cols.append(idx[q]); vals.append(wq)
    M = identity(len(pts), format="csc") - coo_matrix((vals, (rows, cols)), shape=(len(pts), len(pts))).tocsc()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        u = spsolve(M, b)
    if not np.all(np.isfinite(u)) or np.abs(M @ u - b).max() > 1e-8:
        return None, None
    return dict(zip(pts, u)), stencils

In [30]:
def adjoint(stencils, z_star):
    """beta_p = how much of u_hat(z*) comes from point p: solve (I - W)^T beta = e_z*."""
    pts = list(stencils)
    idx = {p: i for i, p in enumerate(pts)}
    rows, cols, vals = [], [], []
    for p, (nbs, w) in stencils.items():
        for q, wq in zip(nbs, w):
            if not is_known(*q):
                rows.append(idx[p]); cols.append(idx[q]); vals.append(wq)
    M = identity(len(pts), format="csc") - coo_matrix((vals, (rows, cols)), shape=(len(pts), len(pts))).tocsc()
    e = np.zeros(len(pts)); e[idx[z_star]] = 1.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        beta = spsolve(M.T.tocsc(), e)
    return None if not np.all(np.isfinite(beta)) else dict(zip(pts, beta))


In [32]:
K_T, R_X, H_TOP = 2, 2, 2      # window depth/width, ceiling above t*

WINDOW = [(dix, dit) for dit in range(-K_T, K_T + 1) for dix in range(-R_X, R_X + 1)
          if (dix, dit) != (0, 0)]

def candidates(cloud, t_top):
    """Empty lattice cells within WINDOW of an existing cloud point, below the ceiling."""
    out = set()
    for (px, pt) in cloud.pts:
        for (dix, dit) in WINDOW:
            qx, qt = px + dix, pt + dit
            if 0 <= qx < nx and 0 <= qt <= t_top and (qx, qt) not in cloud and not is_known(qx, qt):
                out.add((qx, qt))
    return out

def admissible_candidates(cloud, t_top, k=5):
    """Candidates that are geometrically legal AND would get an admissible stencil of their own."""
    return {q for q in candidates(cloud, t_top) if build_stencil(cloud, *q, k=k) is not None}